<a href="https://colab.research.google.com/github/Jyotiii49/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule

I will rank content using two observable signals:

1. **Staleness:** older content gets a higher score because it may need a refresh.
2. **Page-one CTR opportunity:** content ranking in the top 10 with CTR below 0.10 gets an additional point.

Score:
- 2 points if `days_since_last_update >= 91`
- 1 point if `days_since_last_update` is 31–90
- 1 additional point if `avg_position <= 10` and `ctr < 0.10`

Action:
- Score >= 2 → `REFRESH`
- Score = 1 → `REVIEW`
- Score = 0 → `MONITOR`

Reason codes:
- `STALE`
- `LOW_CTR_PAGE1`
- `NONE`

If both conditions apply, the single reason code is `STALE`.

This is a baseline decision-support rule, not a prediction of future performance.

In [6]:
from google.colab import files

uploaded = files.upload()

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")

# Signal 1: staleness
df["stale_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30", "31-90", "91-180", "181+"]
)

stale_check = (
    df.groupby("stale_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          refresh_candidates=("is_initial_refresh_candidate", "sum")
      )
)

stale_check["candidate_rate"] = (
    stale_check["refresh_candidates"] / stale_check["n"]
)

print("SIGNAL 1 — STALENESS")
print(stale_check)

SIGNAL 1 — STALENESS
                  n refresh_candidates candidate_rate
stale_bucket                                         
0-30          20480               6519       0.318311
31-90           175                 89       0.508571
91-180         9171               4759       0.518918
181+            174                 21        0.12069


In [9]:
# Signal 2: CTR vs Position

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[-1, 3, 10, 20, float("inf")],
    labels=["top_3", "4-10", "11-20", "21+"]
)

df["low_ctr_page1"] = (
    (df["avg_position"] <= 10) &
    (df["ctr"] < 0.10)
)

ctr_position_check = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          low_ctr_n=("low_ctr_page1", "sum"),
          ctr_fix_n=("needs_ctr_fix", "sum")
      )
)

ctr_position_check["low_ctr_rate"] = (
    ctr_position_check["low_ctr_n"] / ctr_position_check["n"]
)

ctr_position_check["ctr_fix_rate"] = (
    ctr_position_check["ctr_fix_n"] / ctr_position_check["n"]
)

print("SIGNAL 2 — CTR VS POSITION")
print(ctr_position_check)

SIGNAL 2 — CTR VS POSITION
                     n  low_ctr_n  ctr_fix_n  low_ctr_rate  ctr_fix_rate
position_bucket                                                         
top_3             2346       1870         79      0.797101      0.033674
4-10             11842       4932        458      0.416484      0.038676
11-20             7273          0          0      0.000000      0.000000
21+               8539          0          0      0.000000      0.000000


## 1. My rule and its reason codes

### Observed signals

**Signal 1 — Staleness: MIXED**

The measured refresh-candidate rate was 31.8% for content updated in 0–30 days, 50.9% for 31–90 days, 51.9% for 91–180 days, and 12.1% for 181+ days. Because the rate does not consistently increase with age, the staleness signal is MIXED.

**Signal 2 — CTR vs Position: CONFIRMED**

The low-CTR condition was observed in 79.7% of top-3 rows and 41.6% of rows in positions 4–10. This supports a directional page-one CTR opportunity.

### Baseline rule

I will use a simple score based on observable current/historical signals:

- 2 points when `days_since_last_update >= 91`
- 1 point when `days_since_last_update` is 31–90
- 1 additional point when `avg_position <= 10` and `ctr < 0.10`

Action labels:

- Score >= 2 → `REFRESH`
- Score = 1 → `REVIEW`
- Score = 0 → `MONITOR`

Reason codes:

- `STALE` — content has not been updated recently
- `LOW_CTR_PAGE1` — page-one content has a low CTR
- `NONE` — neither baseline condition applies

This is a directional decision-support baseline, not a prediction or guarantee of future performance.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
# Section 2 — Build the ranked queue

# Calculate staleness score
df["stale_score"] = np.select(
    [
        df["days_since_last_update"] >= 91,
        df["days_since_last_update"] >= 31
    ],
    [2, 1],
    default=0
)

# Calculate page-one CTR opportunity
df["ctr_opportunity_score"] = (
    (df["avg_position"] <= 10) &
    (df["ctr"] < 0.10)
).astype(int)

# Final score
df["action_score"] = (
    df["stale_score"] +
    df["ctr_opportunity_score"]
)

# Action
df["action"] = np.select(
    [
        df["action_score"] >= 2,
        df["action_score"] == 1
    ],
    [
        "REFRESH",
        "REVIEW"
    ],
    default="MONITOR"
)

# One reason code
df["reason_code"] = np.select(
    [
        df["stale_score"] >= 2,
        df["ctr_opportunity_score"] == 1
    ],
    [
        "STALE",
        "LOW_CTR_PAGE1"
    ],
    default="NONE"
)

# Create output folder
import os
os.makedirs("work/outputs", exist_ok=True)

# Rank everything
ranked = df.sort_values(
    by=["action_score", "days_since_last_update", "impressions_90d"],
    ascending=[False, False, False]
).copy()

ranked["rank"] = range(1, len(ranked) + 1)

# Select output columns
ranked_queue = ranked[
    [
        "rank",
        "content_id",
        "action_score",
        "action",
        "reason_code",
        "days_since_last_update",
        "avg_position",
        "ctr",
        "impressions_90d"
    ]
].copy()

# Write CSV
output_path = "work/outputs/baseline_action_score.csv"
ranked_queue.to_csv(output_path, index=False)

print("CSV created:", output_path)
print("Total rows:", len(ranked_queue))

display(ranked_queue.head(10))

CSV created: work/outputs/baseline_action_score.csv
Total rows: 30000


,rank,content_id,action_score,action,reason_code,days_since_last_update,avg_position,ctr,impressions_90d
26242,1,content_55a5b1c46474,3,REFRESH,STALE,373,7.5,0.0,35
24216,2,content_1b4ec72dafd4,3,REFRESH,STALE,372,7.0,0.0,2
8631,3,content_e2b702f4f92b,3,REFRESH,STALE,334,9.3,0.0,30
15608,4,content_06e19c6486b0,3,REFRESH,STALE,334,5.0,0.0,10
21984,5,content_02b0d6e30129,3,REFRESH,STALE,313,6.9,0.0,176
3723,6,content_f488400fca67,3,REFRESH,STALE,305,5.7,0.0,155
8125,7,content_ccf25ed65a99,3,REFRESH,STALE,305,0.0,0.0,2
1147,8,content_ab27c30d81f4,3,REFRESH,STALE,304,8.9,0.0,103
29906,9,content_07ce98c6085a,3,REFRESH,STALE,304,5.3,0.0,85
15052,10,content_7736e6144f3b,3,REFRESH,STALE,304,5.2,0.0,11


## 3. Top-20 review

The top 20 rows are reviewed using the baseline action score.

For each pick, I record:
- the recommended action,
- the single reason code,
- a confidence note based on observed impressions,
- and what could make the recommendation wrong.

These are decision-support recommendations, not guaranteed outcomes.

In [11]:
# Section 3 — Top-20 Review

top20 = ranked_queue.head(20).copy()

# Confidence note based on observed impressions
def confidence_note(impressions):
    if impressions >= 1000:
        return "High: 1,000+ impressions"
    elif impressions >= 100:
        return "Medium: 100–999 impressions"
    else:
        return "Low: <100 impressions"

top20["confidence_note"] = top20["impressions_90d"].apply(
    confidence_note
)

# What could make the recommendation wrong
top20["what_would_make_it_wrong"] = np.where(
    top20["impressions_90d"] < 100,
    "Low traffic may make the CTR/position evidence unstable.",
    "The page may still be performing well despite the baseline signal."
)

top20_review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(top20_review)

,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
26242,1,content_55a5b1c46474,REFRESH,STALE,Low: <100 impressions,Low traffic may make the CTR/position evidence...
24216,2,content_1b4ec72dafd4,REFRESH,STALE,Low: <100 impressions,Low traffic may make the CTR/position evidence...
8631,3,content_e2b702f4f92b,REFRESH,STALE,Low: <100 impressions,Low traffic may make the CTR/position evidence...
15608,4,content_06e19c6486b0,REFRESH,STALE,Low: <100 impressions,Low traffic may make the CTR/position evidence...
21984,5,content_02b0d6e30129,REFRESH,STALE,Medium: 100–999 impressions,The page may still be performing well despite ...
3723,6,content_f488400fca67,REFRESH,STALE,Medium: 100–999 impressions,The page may still be performing well despite ...
8125,7,content_ccf25ed65a99,REFRESH,STALE,Low: <100 impressions,Low traffic may make the CTR/position evidence...
1147,8,content_ab27c30d81f4,REFRESH,STALE,Medium: 100–999 impressions,The page may still be performing well despite ...
29906,9,content_07ce98c6085a,REFRESH,STALE,Low: <100 impressions,Low traffic may make the CTR/position evidence...
15052,10,content_7736e6144f3b,REFRESH,STALE,Low: <100 impressions,Low traffic may make the CTR/position evidence...


## 4. Weak picks + leakage check

The weaker picks are mainly the rows with fewer than 100 observed impressions. Their CTR and position signals may be less stable because the amount of observed traffic is small.

The baseline score uses only `days_since_last_update`, `avg_position`, and `ctr`.

I did not use FlyRank product flags such as `is_quick_win`, `needs_ctr_fix`, `needs_engagement_fix`, `is_declining`, or `is_initial_refresh_candidate` as scoring inputs.

I also did not use future-window or future-label information.

Therefore, this baseline is intended for directional decision-support rather than as a prediction of future performance.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
# Section 4 — Weak picks

weak_picks = top20[
    top20["impressions_90d"] < 100
][[
    "rank",
    "content_id",
    "action_score",
    "action",
    "reason_code",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr"
]]

print("WEAK PICKS — Top-20 rows with fewer than 100 impressions")
display(weak_picks)

WEAK PICKS — Top-20 rows with fewer than 100 impressions


,rank,content_id,action_score,action,reason_code,impressions_90d,days_since_last_update,avg_position,ctr
26242,1,content_55a5b1c46474,3,REFRESH,STALE,35,373,7.5,0.0
24216,2,content_1b4ec72dafd4,3,REFRESH,STALE,2,372,7.0,0.0
8631,3,content_e2b702f4f92b,3,REFRESH,STALE,30,334,9.3,0.0
15608,4,content_06e19c6486b0,3,REFRESH,STALE,10,334,5.0,0.0
8125,7,content_ccf25ed65a99,3,REFRESH,STALE,2,305,0.0,0.0
29906,9,content_07ce98c6085a,3,REFRESH,STALE,85,304,5.3,0.0
15052,10,content_7736e6144f3b,3,REFRESH,STALE,11,304,5.2,0.0
24557,11,content_84d12054c0c0,3,REFRESH,STALE,1,304,8.0,0.0
7222,12,content_8f2c815af658,3,REFRESH,STALE,7,301,6.4,0.0
23619,13,content_24abafed9707,3,REFRESH,STALE,4,231,1.3,0.0


In [13]:
# Leakage check

score_inputs = [
    "days_since_last_update",
    "avg_position",
    "ctr"
]

product_flags = [
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "is_declining",
    "is_initial_refresh_candidate"
]

# Check whether any product flags were used in scoring
leaked_flags = set(score_inputs).intersection(product_flags)

print("Score inputs:")
print(score_inputs)

print("\nProduct flags used in score:")
print(leaked_flags)

if len(leaked_flags) == 0:
    print("\nPASS — No FlyRank product flags leaked into the score.")
else:
    print("\nCHECK — Product flag leakage detected.")

# Check for obvious future-window columns
future_columns = [
    col for col in df.columns
    if any(x in col.lower() for x in [
        "future", "next_30", "next_60", "next_90"
    ])
]

print("\nFuture-window columns found:")
print(future_columns)

if len(future_columns) == 0:
    print("PASS — No obvious future-window columns found.")

print("\nFinal scoring inputs:")
print(score_inputs)

Score inputs:
['days_since_last_update', 'avg_position', 'ctr']

Product flags used in score:
set()

PASS — No FlyRank product flags leaked into the score.

Future-window columns found:
[]
PASS — No obvious future-window columns found.

Final scoring inputs:
['days_since_last_update', 'avg_position', 'ctr']


## Self-check

- [x] Every section is filled with markdown thinking and supporting code.
- [x] Two signals were checked before encoding the rule.
- [x] Staleness verdict: MIXED.
- [x] CTR vs position verdict: CONFIRMED.
- [x] The baseline rule uses observable signals: `days_since_last_update`, `avg_position`, and `ctr`.
- [x] The ranked queue contains 30,000 rows.
- [x] `work/outputs/baseline_action_score.csv` was generated successfully.
- [x] The Top-20 review contains action, reason code, confidence note, and what could make each pick wrong.
- [x] Weak picks were reviewed using observed impression volume.
- [x] No FlyRank product flags leaked into the score.
- [x] No obvious future-window columns were used.
- [x] Claims are presented as directional decision-support, not guaranteed predictions.
- [x] No client names, URLs, or private queries were included.
- [ ] Run the complete notebook from top to bottom with **Runtime → Run all**.
- [ ] Save the executed notebook under `work/notebooks/w04_baseline_score.ipynb`.
- [ ] Commit the notebook to the repository.
- [ ] Submit the repository URL on the assignment card.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.